In [7]:
# Underwriting Issuance Analysis (Notebook Version)

import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path

import plotly.express as px
import plotly.graph_objects as go

from io import BytesIO
import openpyxl

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)


In [8]:
# 1. Configure data source

# Option A: Use default CSV from the data folder
DATA_PATH = Path("data/auto_issuance_synthetic_1year_10000rows.csv")

# Option B: Manually point to another CSV/Excel file
# DATA_PATH = Path("path/to/your/file.csv")
# DATA_PATH = Path("path/to/your/file.xlsx")

print(f"Using data file: {DATA_PATH}")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Data file not found: {DATA_PATH}")

# Helper to load CSV or Excel

def load_data(path: Path) -> pd.DataFrame:
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path, low_memory=False)
    elif path.suffix.lower() in {".xlsx", ".xls"}:
        return pd.read_excel(path, engine="openpyxl")
    else:
        raise ValueError(f"Unsupported file type: {path.suffix}")

original_df = load_data(DATA_PATH)
print("Rows:", len(original_df))
original_df.head()


Using data file: data/auto_issuance_synthetic_1year_10000rows.csv
Rows: 10000


,requestId,opportunityId,ambiguousOpportunityIndicator,processId,requestTypeCode,requestTypeDescription,requestEffectiveDate,statusCode,statusDescription,onHoldReasonCode,onHoldReasonDescription,writeOutReasonCode,writeOutReasonDescription,bgiCode,bgiDescription,lineOfBusinessCode,lineOfBusinessDescription,priorityCode,priorityDescription,accountSystemOfRecordCode,accountSystemOfRecordDescription,salesOfficeCode,salesOfficeDescription,salesRepCode,salesRepDescription,underwritingSegmentCode,underwritingSegmentDescription,regionCode,regionDescription,divisionCode,divisionDescription,targetDateTime,negotiatedTargetDateTime,createDateTime,completedDateTime,endDateTime,receivedDateTime,Aging,standardSLAViolationIndicator,rushSLAViolationIndicator,policyNumber,policyName,policyEffectiveDate,policyExpirationDate,quoteExpirationDate,raterUserIdentificationNumber,raterFullName,assigneeType,localAccountId,accountNumber,midwayCustomerNumber,CCOProspectIdNumber,underwriter,accountAnalyst,accountName,onHoldDatesHistory,Last Hold,offHoldDatesHistory,onHoldReasonCodesHistory,Hold Time,onHoldReasonDescriptionsHistory,Booking Validation HT,EPA-Validate Policy Review HT,Large Audit Variance HT,Other HT,System Issues HT,Underwriting Request HT,Write-Out HT,No of Holds,Types of Case,Booking Validation,EPA-Validate Policy Review,Large Audit Variance,Other,System Issues,Underwriting Request,Write-Out,TAT,TAT Bands,writeOutReasonCodesHistory,writeOutReasonDescriptionsHistory,reRateCode,reRateReason,versions,agentBrokerNum,estimatedAnnualPremium,writeOutCreateDate,writeOutCreator,writeOutReasonCodes,writeOutDescriptions,writeOutRecipientId,writeOutRecipientName,writeOutCCId,writeOutCCName,createdByName,createdById,rpcId,rushReasonDescription,endorsementDescription,underwriterName,accountAnalystName,bureauState,bureauNumber,numberOfLocations,volume,Gross_TAT_Days,Net_TAT_Days,SLA_Status,fieldInError,editErrorNumber,editErrorMessage,ncicIDNumber,underwritingSubsegmentCode,underwritingSubsegmentDescription,receivedInUnderwritingDate,replacementReasonCode,replacementReasonDescription,errorCode,asrRO,asrVIN,modEffectiveDate,revisedMod,revisedModDate,eibIdentifier,ledgerNumber,origin,prelimIndicator,batchRunId,numberOfRisks,numberOfLocations__2,transactionId,editReason,sequence,cancellationReason,subReason,flags,AgentBrokerName,asrRO__2,asrVIN__2,modEffectiveDate__2,revisedModType,revisedModFactor,eibIdentifier__2,ledgerNumber__2,origin__2,prelimIndicator__2,batchRunNumber,numberOfTransactions,numberOfRuns,transactionType,editReason__2,sequenceNumbers,cancellationTypeDescription,reasonForCancellationDescription,subReasonForCancellationDescription,flags__2,AgentBrokerStateCode,AgentBrokerName__2,ManualTransaction,BartVerified,PolicyView,PremiumImpacting,BartTransactionCreatedDate,NumberOfVehicles,NumberOfClaims,WriteOutText,CurrentModType,CurrentModFactor,NumberOfPotentialPolicies,ePolicyB2B,ePolicyNonB2B,TIV
0,REQXAJI0Y6DPBHS,OPPAHXTHV3A3Z,0,PRCMF8MDD4V,CAN,"Endorsement, needs approval, urgent",2025-01-23,HOLD,"On hold, awaiting info",NaN,NaN,NaN,"Write-out reason, follow-up required, add notes",BG2,"Business group, follow-up required, add notes",COMM,"Commercial, fleet",P3,"Low, backlog",LEG,"System of record, follow-up required, add notes",SO16,"Sales office, pending review",SR162,"Sales rep, follow-up required, add notes",STD,"Standard, retail",NE,North East,D1,Division 1,2025-01-27 00:01:34,2025-01-29 00:01:34,2025-01-24 00:01:34,NaN,NaN,2025-01-25 00:01:34,NaN,NaN,NaN,PN48840994,Auto Secure,2025-02-16,2026-02-16,2025-02-13,R02DGTFGQ5,Aditya Roy,Underwriter,LAB94874FR,AC4586084,MC520521,CCO161483,Reyansh Das,Aarav Das,"GreenLeaf Foods, follow-up required, add notes",NaN,NaN,NaN,NaN,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,Underwriting Request,0,0,0,0,0,1,0,NaN,NaN,WO_D,"Write-out reason, needs approval, urgent",RR1,"Re-rate requested, ok",7,AB804314,200.00,2025-11-29 11:28:57,Aarohi Roy,NaN,NaN,NaN,NaN,NaN,NaN,Ishaan Iyer,UCATVAZCC,RPCBLSKRT05,NaN,NaN,Ro

In [9]:
# 2. Helper functions for parsing and calculations

def parse_separated_values(value, separator=None):
    """Parse values that may be separated by ', ' or '|' into a list."""
    if pd.isna(value) or value == "":
        return []

    value = str(value).strip()
    if separator:
        separators = [separator]
    else:
        if "|" in value:
            separators = ["|"]
        elif ", " in value:
            separators = [", "]
        else:
            return [value] if value else []

    result = []
    for sep in separators:
        if sep in value:
            result = [v.strip() for v in value.split(sep) if v.strip()]
            break

    return result if result else ([value] if value else [])


def calculate_aging(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["createDateTime"] = pd.to_datetime(df["createDateTime"], errors="coerce")
    df["completedDateTime"] = pd.to_datetime(df["completedDateTime"], errors="coerce")

    mask = df["completedDateTime"].isna()
    df.loc[mask, "Aging_Days"] = (datetime.now() - df.loc[mask, "createDateTime"]).dt.days
    df.loc[~mask, "Aging_Days"] = np.nan
    return df


def classify_case_type(on_hold_reasons) -> str:
    reasons = parse_separated_values(on_hold_reasons)
    if len(reasons) == 0:
        return "Straight Through"
    elif len(reasons) == 1:
        return "One Touch"
    else:
        return f"Multi Hold ({len(reasons)} touches)"


def calculate_holding_times(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    holding_times_list = []

    for _, row in df.iterrows():
        on_hold_dates = parse_separated_values(row.get("onHoldDatesHistory", ""))
        off_hold_dates = parse_separated_values(row.get("offHoldDatesHistory", ""))

        on_hold_dates_parsed = [
            pd.to_datetime(d, errors="coerce") for d in on_hold_dates
        ]
        on_hold_dates_parsed = [d for d in on_hold_dates_parsed if pd.notna(d)]

        off_hold_dates_parsed = [
            pd.to_datetime(d, errors="coerce") for d in off_hold_dates
        ]
        off_hold_dates_parsed = [d for d in off_hold_dates_parsed if pd.notna(d)]

        hold_times = []
        for i, on_date in enumerate(on_hold_dates_parsed):
            if i < len(off_hold_dates_parsed):
                off_date = off_hold_dates_parsed[i]
                if pd.notna(on_date) and pd.notna(off_date):
                    hold_times.append((off_date - on_date).days)
            else:
                if pd.notna(on_date):
                    hold_times.append((datetime.now() - on_date).days)

        holding_times_list.append(hold_times)

    df["HoldingTimes"] = holding_times_list
    df["TotalHoldingTime"] = df["HoldingTimes"].apply(lambda x: sum(x) if x else 0)
    df["NumberOfTouches"] = df["HoldingTimes"].apply(lambda x: len(x) if x else 0)
    return df


def calculate_tat(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["createDateTime"] = pd.to_datetime(df["createDateTime"], errors="coerce")
    df["completedDateTime"] = pd.to_datetime(df["completedDateTime"], errors="coerce")

    mask = df["completedDateTime"].notna()
    df.loc[mask, "TAT_Days"] = (
        df.loc[mask, "completedDateTime"] - df.loc[mask, "createDateTime"]
    ).dt.days
    df.loc[~mask, "TAT_Days"] = np.nan
    return df


def create_tat_bucket(tat_days):
    if pd.isna(tat_days):
        return None
    if tat_days <= 5:
        return "0-5 days"
    elif tat_days <= 7:
        return "5-7 days"
    else:
        return "7+ days"


In [10]:
# 3. Process data (mirrors Streamlit app logic)

# Work on a copy for processing
df = original_df.copy()

# Basic calculations
df = calculate_aging(df)

df["CaseType"] = df["onHoldReasonDescriptionsHistory"].apply(classify_case_type)

df = calculate_holding_times(df)

df = calculate_tat(df)

df["TAT_Bucket"] = df["TAT_Days"].apply(create_tat_bucket)

# Month / seasonality
if "createDateTime" in df.columns:
    df["createDateTime"] = pd.to_datetime(df["createDateTime"], errors="coerce")
    df["Month"] = df["createDateTime"].dt.to_period("M")
    df["Month_Str"] = df["Month"].astype(str)

# Hold category (Straight / Single / Multi)
df["HoldCategory"] = df["CaseType"].apply(
    lambda x: "Straight Through" if x == "Straight Through"
    else "Single Hold" if x == "One Touch"
    else "Multi Hold"
)

print("Processed rows:", len(df))
df.head()


Processed rows: 10000


,requestId,opportunityId,ambiguousOpportunityIndicator,processId,requestTypeCode,requestTypeDescription,requestEffectiveDate,statusCode,statusDescription,onHoldReasonCode,onHoldReasonDescription,writeOutReasonCode,writeOutReasonDescription,bgiCode,bgiDescription,lineOfBusinessCode,lineOfBusinessDescription,priorityCode,priorityDescription,accountSystemOfRecordCode,accountSystemOfRecordDescription,salesOfficeCode,salesOfficeDescription,salesRepCode,salesRepDescription,underwritingSegmentCode,underwritingSegmentDescription,regionCode,regionDescription,divisionCode,divisionDescription,targetDateTime,negotiatedTargetDateTime,createDateTime,completedDateTime,endDateTime,receivedDateTime,Aging,standardSLAViolationIndicator,rushSLAViolationIndicator,policyNumber,policyName,policyEffectiveDate,policyExpirationDate,quoteExpirationDate,raterUserIdentificationNumber,raterFullName,assigneeType,localAccountId,accountNumber,midwayCustomerNumber,CCOProspectIdNumber,underwriter,accountAnalyst,accountName,onHoldDatesHistory,Last Hold,offHoldDatesHistory,onHoldReasonCodesHistory,Hold Time,onHoldReasonDescriptionsHistory,Booking Validation HT,EPA-Validate Policy Review HT,Large Audit Variance HT,Other HT,System Issues HT,Underwriting Request HT,Write-Out HT,No of Holds,Types of Case,Booking Validation,EPA-Validate Policy Review,Large Audit Variance,Other,System Issues,Underwriting Request,Write-Out,TAT,TAT Bands,writeOutReasonCodesHistory,writeOutReasonDescriptionsHistory,reRateCode,reRateReason,versions,agentBrokerNum,estimatedAnnualPremium,writeOutCreateDate,writeOutCreator,writeOutReasonCodes,writeOutDescriptions,writeOutRecipientId,writeOutRecipientName,writeOutCCId,writeOutCCName,createdByName,createdById,rpcId,rushReasonDescription,endorsementDescription,underwriterName,accountAnalystName,bureauState,bureauNumber,numberOfLocations,volume,Gross_TAT_Days,Net_TAT_Days,SLA_Status,fieldInError,editErrorNumber,editErrorMessage,ncicIDNumber,underwritingSubsegmentCode,underwritingSubsegmentDescription,receivedInUnderwritingDate,replacementReasonCode,replacementReasonDescription,errorCode,asrRO,asrVIN,modEffectiveDate,revisedMod,revisedModDate,eibIdentifier,ledgerNumber,origin,prelimIndicator,batchRunId,numberOfRisks,numberOfLocations__2,transactionId,editReason,sequence,cancellationReason,subReason,flags,AgentBrokerName,asrRO__2,asrVIN__2,modEffectiveDate__2,revisedModType,revisedModFactor,eibIdentifier__2,ledgerNumber__2,origin__2,prelimIndicator__2,batchRunNumber,numberOfTransactions,numberOfRuns,transactionType,editReason__2,sequenceNumbers,cancellationTypeDescription,reasonForCancellationDescription,subReasonForCancellationDescription,flags__2,AgentBrokerStateCode,AgentBrokerName__2,ManualTransaction,BartVerified,PolicyView,PremiumImpacting,BartTransactionCreatedDate,NumberOfVehicles,NumberOfClaims,WriteOutText,CurrentModType,CurrentModFactor,NumberOfPotentialPolicies,ePolicyB2B,ePolicyNonB2B,TIV,Aging_Days,CaseType,HoldingTimes,TotalHoldingTime,NumberOfTouches,TAT_Days,TAT_Bucket,Month,Month_Str,HoldCategory
0,REQXAJI0Y6DPBHS,OPPAHXTHV3A3Z,0,PRCMF8MDD4V,CAN,"Endorsement, needs approval, urgent",2025-01-23,HOLD,"On hold, awaiting info",NaN,NaN,NaN,"Write-out reason, follow-up required, add notes",BG2,"Business group, follow-up required, add notes",COMM,"Commercial, fleet",P3,"Low, backlog",LEG,"System of record, follow-up required, add notes",SO16,"Sales office, pending review",SR162,"Sales rep, follow-up required, add notes",STD,"Standard, retail",NE,North East,D1,Division 1,2025-01-27 00:01:34,2025-01-29 00:01:34,2025-01-24 00:01:34,NaT,NaN,2025-01-25 00:01:34,NaN,NaN,NaN,PN48840994,Auto Secure,2025-02-16,2026-02-16,2025-02-13,R02DGTFGQ5,Aditya Roy,Underwriter,LAB94874FR,AC4586084,MC520521,CCO161483,Reyansh Das,Aarav Das,"GreenLeaf Foods, follow-up required, add notes",NaN,NaN,NaN,NaN,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,Underwriting Request,0,0,0,0,0,1,0,NaN,NaN,WO_D,"Write-out reason, needs approval, urgent",RR1,"Re-rate requested, ok",7,

In [11]:
# 4. Key metrics and distributions

# Status counts
if "statusDescription" in df.columns:
    status_counts = df["statusDescription"].value_counts()
    display(status_counts.to_frame("Count"))

# Aging (non-completed)
aging_data = df[df["Aging_Days"].notna()].copy()
print("Non-completed cases (for aging):", len(aging_data))

# Case type distribution
case_type_counts = df["CaseType"].value_counts()
print("\nCase type counts:")
display(case_type_counts.to_frame("Count"))

# Straight / one-touch / multi-hold subsets
straight_through = df[df["CaseType"] == "Straight Through"]
one_touch = df[df["CaseType"] == "One Touch"]
multi_hold = df[df["CaseType"].str.contains("Multi Hold", na=False)]

print("\nStraight Through:", len(straight_through))
print("One Touch:", len(one_touch))
print("Multi Hold:", len(multi_hold))

# Holding time summary
holding_data = df[df["TotalHoldingTime"] > 0].copy()
print("\nCases with holding time:", len(holding_data))

# TAT data
tat_data = df[df["TAT_Days"].notna()].copy()
print("Cases with TAT:", len(tat_data))

# Median TAT overall
median_tat = tat_data["TAT_Days"].median()
print("\nMedian TAT (days):", median_tat)


,Count
statusDescription,
"In progress, processing",3514
"New, created",2500
"Completed, issued",1955
"On hold, awaiting info",1533
"Cancelled, closed",498


Non-completed cases (for aging): 7547

Case type counts:


,Count
CaseType,
Straight Through,6519
Multi Hold (2 touches),3066
Multi Hold (3 touches),244
One Touch,171



Straight Through: 6519
One Touch: 171
Multi Hold: 3310

Cases with holding time: 2979
Cases with TAT: 2453

Median TAT (days): 3.0


In [6]:
# 5. Visualizations (Plotly)

# Status count bar chart
if "statusDescription" in df.columns:
    fig_status = px.bar(
        status_counts.reset_index(),
        x="index",
        y="statusDescription",
        labels={"index": "Status", "statusDescription": "Count"},
        title="Status Count"
    )
    fig_status.update_xaxes(tickangle=45)
    fig_status.show()

# Aging distribution
if not aging_data.empty:
    fig_aging = px.histogram(
        aging_data,
        x="Aging_Days",
        nbins=30,
        title="Distribution of Aging Days",
        labels={"Aging_Days": "Aging (Days)", "count": "Frequency"}
    )
    fig_aging.show()

# Case type distribution (pie)
fig_case_type = px.pie(
    case_type_counts.reset_index(),
    values="CaseType",
    names="index",
    title="Case Type Distribution"
)
fig_case_type.show()

# Holding time distribution
if not holding_data.empty:
    fig_holding = px.histogram(
        holding_data,
        x="TotalHoldingTime",
        nbins=30,
        title="Distribution of Total Holding Time (Days)",
        labels={"TotalHoldingTime": "Holding Time (Days)", "count": "Frequency"}
    )
    fig_holding.show()

# TAT distribution
if not tat_data.empty:
    fig_tat = px.histogram(
        tat_data,
        x="TAT_Days",
        nbins=30,
        title="Distribution of TAT (Days)",
        labels={"TAT_Days": "TAT (Days)", "count": "Frequency"}
    )
    fig_tat.show()

# TAT buckets
tat_bucket_counts = df["TAT_Bucket"].value_counts()
fig_buckets = px.bar(
    tat_bucket_counts.reset_index(),
    x="index",
    y="TAT_Bucket",
    labels={"index": "TAT Bucket", "TAT_Bucket": "Count"},
    title="TAT Bucket Distribution"
)
fig_buckets.show()

# Median TAT by Case Type
tat_by_case_type = df.groupby("CaseType")["TAT_Days"].median().sort_values(ascending=False)
fig_tat_case = px.bar(
    tat_by_case_type.reset_index(),
    x="CaseType",
    y="TAT_Days",
    labels={"CaseType": "Case Type", "TAT_Days": "Median TAT (Days)"},
    title="Median TAT by Case Type"
)
fig_tat_case.update_xaxes(tickangle=45)
fig_tat_case.show()

# Median TAT by HoldCategory and TAT_Bucket
tat_by_hold_bucket_raw = df.groupby(["HoldCategory", "TAT_Bucket"])['TAT_Days'].median().reset_index()
if not tat_by_hold_bucket_raw.empty:
    tat_by_hold_bucket = tat_by_hold_bucket_raw.pivot(index="TAT_Bucket", columns="HoldCategory", values="TAT_Days")
    fig_tat_hold_bucket = px.bar(
        tat_by_hold_bucket.reset_index(),
        x="TAT_Bucket",
        y=[c for c in tat_by_hold_bucket.columns if c != "TAT_Bucket"],
        barmode="group",
        title="Median TAT by Hold Category and TAT Bucket",
        labels={"value": "Median TAT (Days)", "TAT_Bucket": "TAT Bucket"}
    )
    fig_tat_hold_bucket.show()

# Median TAT by HoldCategory overall
tat_by_hold_category = df.groupby("HoldCategory")["TAT_Days"].median().sort_values(ascending=False)
print("\nMedian TAT by Hold Category:")
display(tat_by_hold_category.to_frame("Median_TAT_Days"))


ValueError: Value of 'x' is not the name of a column in 'data_frame'. Expected one of ['statusDescription', 'count'] but received: index
 To use the index, pass it in directly as `df.index`.

In [ ]:
# 6. Drill-down analysis (without Streamlit UI)

# Example: change these parameters to explore different segments
DRILL_MODE = "By TAT Bucket"  # options: "Straight Through", "Multi Hold", "By TAT Bucket", "By Number of Touches"
DRILL_TAT_BUCKET = "0-5 days"  # used when DRILL_MODE == "By TAT Bucket"
DRILL_TOUCHES = 1              # used when DRILL_MODE == "By Number of Touches"

if DRILL_MODE == "Straight Through":
    drill_data = df[df["CaseType"] == "Straight Through"].copy()
elif DRILL_MODE == "Multi Hold":
    drill_data = df[df["CaseType"].str.contains("Multi Hold", na=False)].copy()
elif DRILL_MODE == "By TAT Bucket":
    drill_data = df[df["TAT_Bucket"] == DRILL_TAT_BUCKET].copy()
elif DRILL_MODE == "By Number of Touches":
    drill_data = df[df["NumberOfTouches"] == DRILL_TOUCHES].copy()
else:
    drill_data = df.copy()

print(f"Drill mode: {DRILL_MODE}, rows: {len(drill_data)}")

if not drill_data.empty:
    # Top 10 request types
    if "requestTypeDescription" in drill_data.columns:
        top_request_types = drill_data["requestTypeDescription"].value_counts().head(10)
        print("\nTop 10 Request Types:")
        display(top_request_types.to_frame("Count"))

        fig_req = px.bar(
            top_request_types.reset_index(),
            x="requestTypeDescription",
            y="count",
            labels={"requestTypeDescription": "Request Type", "count": "Count"},
            title="Top 10 Request Types (Drill-down)",
        )
        fig_req.update_xaxes(tickangle=45)
        fig_req.show()

    # Top 10 hold reasons
    if "onHoldReasonDescriptionsHistory" in drill_data.columns:
        all_hold_reasons = []
        for reasons in drill_data["onHoldReasonDescriptionsHistory"].apply(parse_separated_values):
            all_hold_reasons.extend(reasons)

        if all_hold_reasons:
            hold_reason_counts = pd.Series(all_hold_reasons).value_counts().head(10)
            print("\nTop 10 Hold Reasons:")
            display(hold_reason_counts.to_frame("Count"))

            fig_hold = px.bar(
                hold_reason_counts.reset_index(),
                x="onHoldReasonDescriptionsHistory",
                y="count",
                labels={"onHoldReasonDescriptionsHistory": "Hold Reason", "count": "Count"},
                title="Top 10 Hold Reasons (Drill-down)",
            )
            fig_hold.update_xaxes(tickangle=45)
            fig_hold.show()
else:
    print("No rows for selected drill-down.")


In [ ]:
# 7. Seasonality and location/vehicle analysis

# Seasonality by month
if "Month_Str" in df.columns:
    monthly_stats = df.groupby("Month_Str").agg({
        "requestId": "count",
        "completedDateTime": lambda x: x.notna().sum(),
        "TAT_Days": "median",
        "TotalHoldingTime": "median",
        "NumberOfTouches": "mean",
    }).rename(columns={
        "requestId": "Total_Cases",
        "completedDateTime": "Completed_Cases",
        "TAT_Days": "Median_TAT",
        "TotalHoldingTime": "Median_Holding_Time",
        "NumberOfTouches": "Avg_Touches",
    })

    monthly_stats["Completion_Rate"] = (
        monthly_stats["Completed_Cases"] / monthly_stats["Total_Cases"] * 100
    ).round(2)

    holding_rate = df.groupby("Month_Str")["TotalHoldingTime"].apply(
        lambda x: (x > 0).sum() / len(x) * 100
    ).round(2)
    monthly_stats["Holding_Rate"] = holding_rate

    print("Monthly statistics:")
    display(monthly_stats)

    fig_completion = px.line(
        monthly_stats.reset_index(),
        x="Month_Str",
        y="Completion_Rate",
        title="Monthly Completion Rate (%)",
        labels={"Month_Str": "Month", "Completion_Rate": "Completion Rate (%)"},
    )
    fig_completion.update_xaxes(tickangle=45)
    fig_completion.show()

    fig_holding_rate = px.line(
        monthly_stats.reset_index(),
        x="Month_Str",
        y="Holding_Rate",
        title="Monthly Holding Rate (%)",
        labels={"Month_Str": "Month", "Holding_Rate": "Holding Rate (%)"},
    )
    fig_holding_rate.update_xaxes(tickangle=45)
    fig_holding_rate.show()

# Location / vehicle analysis
if "numberOfLocations" in df.columns and "NumberOfVehicles" in df.columns:
    lv_data = df[(df["numberOfLocations"].notna()) & (df["NumberOfVehicles"].notna())].copy()
    if not lv_data.empty:
        location_analysis = lv_data.groupby("numberOfLocations").agg({
            "TAT_Days": "median",
            "NumberOfTouches": "mean",
            "TotalHoldingTime": "median",
            "requestId": "count",
        }).rename(columns={
            "TAT_Days": "Median_TAT",
            "NumberOfTouches": "Avg_Touches",
            "TotalHoldingTime": "Median_Holding_Time",
            "requestId": "Count",
        })

        print("\nLocation analysis:")
        display(location_analysis)

        fig_loc = px.bar(
            location_analysis.reset_index(),
            x="numberOfLocations",
            y="Median_TAT",
            title="Median TAT by Number of Locations",
            labels={"numberOfLocations": "Number of Locations", "Median_TAT": "Median TAT (Days)"},
        )
        fig_loc.show()

        vehicle_analysis = lv_data.groupby("NumberOfVehicles").agg({
            "TAT_Days": "median",
            "NumberOfTouches": "mean",
            "TotalHoldingTime": "median",
            "requestId": "count",
        }).rename(columns={
            "TAT_Days": "Median_TAT",
            "NumberOfTouches": "Avg_Touches",
            "TotalHoldingTime": "Median_Holding_Time",
            "requestId": "Count",
        })

        print("\nVehicle analysis:")
        display(vehicle_analysis)

        fig_veh = px.bar(
            vehicle_analysis.reset_index(),
            x="NumberOfVehicles",
            y="Median_TAT",
            title="Median TAT by Number of Vehicles",
            labels={"NumberOfVehicles": "Number of Vehicles", "Median_TAT": "Median TAT (Days)"},
        )
        fig_veh.show()

        # Correlations
        fig_corr_touches = px.scatter(
            lv_data,
            x="NumberOfVehicles",
            y="NumberOfTouches",
            trendline="ols",
            title="Number of Vehicles vs Number of Touches",
        )
        fig_corr_touches.show()

        fig_corr_hold = px.scatter(
            lv_data,
            x="NumberOfVehicles",
            y="TotalHoldingTime",
            trendline="ols",
            title="Number of Vehicles vs Holding Time",
        )
        fig_corr_hold.show()


In [ ]:
# 8. Holding time statistics (median highest/lowest etc.)

if not holding_data.empty:
    all_holding_times = []
    for times in holding_data["HoldingTimes"]:
        all_holding_times.extend(times)

    if all_holding_times:
        holding_times_series = pd.Series(all_holding_times)
        highest_per_case = holding_data["HoldingTimes"].apply(lambda x: max(x) if x else 0)
        lowest_per_case = holding_data["HoldingTimes"].apply(lambda x: min(x) if x else 0)

        summary = {
            "Median_Holding_Time_All": holding_times_series.median(),
            "Median_of_Highest_Holding_Time": highest_per_case.median(),
            "Median_of_Lowest_Holding_Time": lowest_per_case.median(),
            "Highest_Holding_Time_Overall": holding_times_series.max(),
            "Lowest_Holding_Time_Overall": holding_times_series.min(),
            "Average_Number_of_Touches": holding_data["NumberOfTouches"].mean(),
        }

        print("Holding time statistics:")
        display(pd.Series(summary))

        fig_holding_dist = px.histogram(
            holding_times_series,
            nbins=30,
            title="Distribution of Individual Holding Times",
            labels={"value": "Holding Time (Days)", "count": "Frequency"},
        )
        fig_holding_dist.show()

        fig_high_low = go.Figure()
        fig_high_low.add_trace(go.Box(y=highest_per_case, name="Highest per Case", boxmean="sd"))
        fig_high_low.add_trace(go.Box(y=lowest_per_case, name="Lowest per Case", boxmean="sd"))
        fig_high_low.update_layout(
            title="Highest vs Lowest Holding Times per Case",
            yaxis_title="Holding Time (Days)",
        )
        fig_high_low.show()


In [ ]:
# 9. Export all results to Excel (downloadable file)

OUTPUT_PATH = Path("underwriting_report_notebook.xlsx")

with pd.ExcelWriter(OUTPUT_PATH, engine="openpyxl") as writer:
    # Summary sheet (similar to Streamlit)
    all_holding_times_flat = []
    if not holding_data.empty:
        for times in holding_data["HoldingTimes"]:
            all_holding_times_flat.extend(times)

    highest_per_case = (
        holding_data["HoldingTimes"].apply(lambda x: max(x) if x else 0)
        if not holding_data.empty
        else pd.Series(dtype=float)
    )
    lowest_per_case = (
        holding_data["HoldingTimes"].apply(lambda x: min(x) if x else 0)
        if not holding_data.empty
        else pd.Series(dtype=float)
    )

    summary_data = {
        "Metric": [
            "Total Cases",
            "Completed Cases",
            "Median TAT (days)",
            "Median Aging (days)",
            "Straight Through Cases",
            "One Touch Cases",
            "Multi Hold Cases",
            "Median Holding Time (All)",
            "Median of Highest Holding Time",
            "Median of Lowest Holding Time",
            "Average Number of Touches",
            "Highest Holding Time (Overall)",
            "Lowest Holding Time (Overall)",
        ],
        "Value": [
            len(df),
            df["completedDateTime"].notna().sum() if "completedDateTime" in df.columns else 0,
            df["TAT_Days"].median() if "TAT_Days" in df.columns and not df["TAT_Days"].isna().all() else 0,
            df["Aging_Days"].median() if "Aging_Days" in df.columns and not df["Aging_Days"].isna().all() else 0,
            len(straight_through),
            len(one_touch),
            len(multi_hold),
            pd.Series(all_holding_times_flat).median() if all_holding_times_flat else 0,
            highest_per_case.median() if not highest_per_case.empty else 0,
            lowest_per_case.median() if not lowest_per_case.empty else 0,
            holding_data["NumberOfTouches"].mean() if not holding_data.empty else 0,
            max(all_holding_times_flat) if all_holding_times_flat else 0,
            min(all_holding_times_flat) if all_holding_times_flat else 0,
        ],
    }
    pd.DataFrame(summary_data).to_excel(writer, sheet_name="Summary", index=False)

    # Status counts
    if "status_counts" in globals():
        sc_df = status_counts.to_frame("Count").reset_index().rename(columns={"index": "Status"})
        sc_df.to_excel(writer, sheet_name="Status_Counts", index=False)

    # Case type classification
    ctc_df = case_type_counts.to_frame("Count").reset_index().rename(columns={"index": "Case_Type"})
    ctc_df.to_excel(writer, sheet_name="Case_Type_Classification", index=False)

    # TAT buckets
    tbc_df = tat_bucket_counts.to_frame("Count").reset_index().rename(columns={"index": "TAT_Bucket"})
    tbc_df.to_excel(writer, sheet_name="TAT_Buckets", index=False)

    # Median TAT by Case Type
    tat_by_case_type_df = tat_by_case_type.reset_index().rename(columns={"TAT_Days": "Median_TAT_Days"})
    tat_by_case_type_df.to_excel(writer, sheet_name="Median_TAT_by_CaseType", index=False)

    # Median TAT by Hold Category and TAT Bucket
    if "tat_by_hold_bucket_raw" in globals() and not tat_by_hold_bucket_raw.empty:
        tat_by_hold_bucket = tat_by_hold_bucket_raw.pivot(
            index="TAT_Bucket", columns="HoldCategory", values="TAT_Days"
        )
        tat_by_hold_bucket.to_excel(writer, sheet_name="Median_TAT_by_Hold_Bucket")

    # Median TAT by Hold Category
    tat_by_hold_category_df = tat_by_hold_category.to_frame("Median_TAT_Days").reset_index().rename(
        columns={"index": "Hold_Category"}
    )
    tat_by_hold_category_df.to_excel(writer, sheet_name="Median_TAT_by_HoldCategory", index=False)

    # Monthly statistics
    if "monthly_stats" in globals():
        monthly_stats.to_excel(writer, sheet_name="Monthly_Statistics")

    # Location / Vehicle analysis
    if "location_analysis" in globals():
        location_analysis.to_excel(writer, sheet_name="Location_Analysis")
    if "vehicle_analysis" in globals():
        vehicle_analysis.to_excel(writer, sheet_name="Vehicle_Analysis")

    # Holding Time Stats
    if all_holding_times_flat:
        holding_stats = {
            "Metric": [
                "Median Holding Time (All)",
                "Median of Highest Holding Time",
                "Median of Lowest Holding Time",
                "Highest Holding Time (Overall)",
                "Lowest Holding Time (Overall)",
                "Average Number of Touches",
            ],
            "Value": [
                pd.Series(all_holding_times_flat).median(),
                highest_per_case.median() if not highest_per_case.empty else 0,
                lowest_per_case.median() if not lowest_per_case.empty else 0,
                max(all_holding_times_flat),
                min(all_holding_times_flat),
                holding_data["NumberOfTouches"].mean() if not holding_data.empty else 0,
            ],
        }
        pd.DataFrame(holding_stats).to_excel(writer, sheet_name="Holding_Time_Stats", index=False)

    # Original data
    original_df.to_excel(writer, sheet_name="Original_Data", index=False)

    # Processed data (essential columns)
    essential_cols = [
        "requestId",
        "statusDescription",
        "createDateTime",
        "completedDateTime",
        "onHoldReasonDescriptionsHistory",
        "onHoldDatesHistory",
        "offHoldDatesHistory",
        "requestTypeDescription",
        "onHoldReasonDescription",
        "numberOfLocations",
        "NumberOfVehicles",
        "CaseType",
        "HoldCategory",
        "Aging_Days",
        "TAT_Days",
        "TAT_Bucket",
        "TotalHoldingTime",
        "NumberOfTouches",
        "Month_Str",
    ]
    available_cols = [c for c in essential_cols if c in df.columns]
    df[available_cols].to_excel(writer, sheet_name="Processed_Data", index=False)

print(f"Excel report written to: {OUTPUT_PATH.resolve()}")
